In [ ]:
import pandas as pd
import numpy as np
import json
import os
import calendar
from datetime import datetime, date

# =============================================================
# ██████╗  █████╗ ██████╗  █████╗ ███╗   ███╗███████╗
# ██╔══██╗██╔══██╗██╔══██╗██╔══██╗████╗ ████║██╔════╝
# ██████╔╝███████║██████╔╝███████║██╔████╔██║███████╗
# ██╔═══╝ ██╔══██║██╔══██╗██╔══██║██║╚██╔╝██║╚════██║
# ██║     ██║  ██║██║  ██║██║  ██║██║ ╚═╝ ██║███████║
# ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═╝╚═╝     ╚═╝╚══════╝
#
#  Smart APS V5  —  Indent-Based Planning
#  Logic  : today_target = max(0, daily_indent - inventory)
#  daily_indent = monthly_indent / working_days_in_month
#  working_days = calendar days in month  −  number of Sundays
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS  (change these every morning)
# -------------------------------------------------------------
# PLANNING_DATE : today's date
# INDENT_MONTH  : the month whose indent column to use
#                 (usually the current month)
#
# Working days are calculated automatically:
#   working_days = total days in month − number of Sundays
# No manual entry needed — just update the two dates below.
# =============================================================

PLANNING_DATE = date(2026, 3, 14)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS  = 22       # total machine hours per shift
MIN_RUN_HOURS    = 4        # minimum block per part on a machine
TARGET_DAYS_INV  = 3        # ideal inventory buffer (days)
MACHINE_STATE_FILE = "machine_state.json"

# Priority weights
W_COVERAGE = 2.0
W_URGENCY  = 3.0
W_RISK     = 1.5

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS CALCULATION
# -------------------------------------------------------------
# working_days = calendar days in INDENT_MONTH
#                minus the number of Sundays in that month
# Example: March 2026 has 31 days and 5 Sundays → 26 working days
# =============================================================

def compute_working_days(ref_date):
    """
    Return (working_days, total_days, sunday_count) for the month
    that contains ref_date.  Sunday = weekday() == 6.
    """
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]   # total days in month

    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    working = total - sundays
    return working, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*62}")
print(f"  Smart APS V5  —  Indent-Based Planning")
print(f"  Planning date  : {PLANNING_DATE}")
print(f"  Indent month   : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Total days     : {TOTAL_DAYS}")
print(f"  Sundays        : {SUNDAY_COUNT}")
print(f"  Working days   : {WORKING_DAYS}  (used as divisor)")
print(f"  Daily target   : monthly_indent / {WORKING_DAYS}")
print(f"  Today target   : max(0, daily_indent - inventory)")
print(f"{'='*62}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
stats        = pd.read_excel(book_path,       sheet_name="Sheet2")
hz_parts_raw = pd.read_excel(book_path,       sheet_name="HZ")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
hz_matrix    = pd.read_excel(matrix_path,     sheet_name="HZ_Matrix")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
hz_co_raw    = pd.read_excel(changeover_path, sheet_name="HZ_Changeover")
vt_co_raw    = pd.read_excel(changeover_path, sheet_name="VT_Changeover")

# =============================================================
# SECTION 6 — MERGE & PRODUCTION RATE
# =============================================================

# Combine HZ and VT part sheets to get inventory and rate
# (stats sheet carries Cycle time, cavity, Inventory on 24th)
hz_combined = hz_parts_raw.copy()
vt_combined = vt_parts_raw.copy()

# We need a unified lookup for rate and inventory.
# Merge both part sheets with stats on Part column.
def find_col(df, name, sheet):
    """Case-insensitive column finder."""
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(f"Column '{name}' not found in sheet '{sheet}'. "
                         f"Available: {list(df.columns)}")
    return match

# stats has "Part" and cycle/cavity/inventory columns
stats_part_col = find_col(stats, "Part", "Sheet2")
data = pd.concat([
    stats.merge(hz_parts_raw.rename(columns={
        find_col(hz_parts_raw, "Part", "HZ"): "Material"
    }), left_on=stats_part_col, right_on="Material", how="inner"),
    stats.merge(vt_parts_raw.rename(columns={
        find_col(vt_parts_raw, "Part", "VT"): "Material"
    }), left_on=stats_part_col, right_on="Material", how="inner"),
], ignore_index=True).drop_duplicates(subset="Material")

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()].copy()

# =============================================================
# SECTION 7 — BUILD LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        k: (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
    }

inventory = safe_dict(data, "Material", "Inventory on 24th")
rate      = safe_dict(data, "Material", "Rate")

# =============================================================
# SECTION 7A — READ INDENT FROM HZ / VT SHEETS
# -------------------------------------------------------------
# Both HZ and VT sheets must have:
#   • A "Part"   column  (case-insensitive)
#   • An "Indent" column (case-insensitive)
#
# indent_monthly  : {part → monthly indent qty}
# indent_daily    : {part → monthly indent / working_days}
# today_target    : {part → max(0, daily_indent - inventory)}
# =============================================================

def read_indent_from_sheet(df, sheet_name):
    """
    Extract {part: monthly_indent} from a Book1 HZ or VT sheet.
    Column names are matched case-insensitively.
    Returns empty dict (with warning) if either column is missing.
    """
    result = {}

    part_col   = next((c for c in df.columns if str(c).strip().lower() == "part"),   None)
    indent_col = next((c for c in df.columns if str(c).strip().lower() == "indent"), None)

    if part_col is None:
        print(f"  WARNING: 'Part' column not found in Book1 sheet '{sheet_name}'")
        print(f"           Available columns: {list(df.columns)}")
        return result
    if indent_col is None:
        print(f"  WARNING: 'Indent' column not found in Book1 sheet '{sheet_name}'")
        print(f"           Available columns: {list(df.columns)}")
        return result

    print(f"  Book1 '{sheet_name}' — Part col: '{part_col}'  Indent col: '{indent_col}'")

    for _, row in df.iterrows():
        part = row[part_col]
        val  = row[indent_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        indent_qty = float(val) if pd.notna(val) else 0.0
        result[str(part).strip()] = indent_qty

    return result

print("\nReading indent data...")
hz_indent_raw = read_indent_from_sheet(hz_parts_raw, "HZ")
vt_indent_raw = read_indent_from_sheet(vt_parts_raw, "VT")

# Merge — VT takes precedence if part appears in both
indent_monthly = {**hz_indent_raw, **vt_indent_raw}

# Daily indent = monthly / working days
indent_daily = {
    part: round(qty / WORKING_DAYS, 4)
    for part, qty in indent_monthly.items()
}

# Today's target = max(0, daily_indent - current_inventory)
today_target_qty = {
    part: max(0.0, indent_daily.get(part, 0.0) - inventory.get(part, 0.0))
    for part in indent_monthly
}

# Print summary
parts_with_indent    = sum(1 for v in indent_monthly.values() if v > 0)
parts_zero_indent    = sum(1 for v in indent_monthly.values() if v == 0)
parts_inv_sufficient = sum(1 for p, v in today_target_qty.items() if v == 0
                           and indent_monthly.get(p, 0) > 0)

print(f"\n  Indent summary:")
print(f"    Parts with non-zero indent : {parts_with_indent}")
print(f"    Parts with zero indent     : {parts_zero_indent}")
print(f"    Parts where inv covers daily target (no production needed today): "
      f"{parts_inv_sufficient}")

# =============================================================
# SECTION 7B — CHANGEOVER TIMES PER MACHINE
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

hz_changeover = build_changeover_dict(hz_co_raw)
vt_changeover = build_changeover_dict(vt_co_raw)

DEFAULT_CHANGEOVER_HRS = 40 / 60.0

print(f"\n  HZ changeover times loaded: {len(hz_changeover)} machines")
for m, h in hz_changeover.items():
    print(f"    {m:25s} → {h*60:.0f} min ({h:.3f} h)")
print(f"\n  VT changeover times loaded: {len(vt_changeover)} machines")
for m, h in vt_changeover.items():
    print(f"    {m:25s} → {h*60:.0f} min ({h:.3f} h)")

# =============================================================
# SECTION 7C — PART CATEGORY  (Runner / Repeater / Stranger)
# =============================================================

def build_category_from_sheet(df, sheet_name):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"),     None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)

    if part_col is None:
        print(f"  WARNING: 'Part' column not found in Book1 sheet '{sheet_name}'")
        return cat
    if cat_col is None:
        print(f"  WARNING: 'Category' column not found in Book1 sheet '{sheet_name}'")
        return cat

    print(f"  Book1 '{sheet_name}' — Part col: '{part_col}'  Category col: '{cat_col}'")

    for _, row in df.iterrows():
        part = row[part_col]
        val  = str(row[cat_col]).strip() if pd.notna(row[cat_col]) else "Stranger"
        if pd.isna(part) or str(part).strip() == "":
            continue
        if val.lower() in ("runner", "repeater", "stranger"):
            val = val.capitalize()
        cat[str(part).strip()] = val
    return cat

hz_cat = build_category_from_sheet(hz_parts_raw, "HZ")
vt_cat = build_category_from_sheet(vt_parts_raw, "VT")

part_category  = {**hz_cat, **vt_cat}
runner_count   = sum(1 for v in part_category.values() if v == "Runner")
repeater_count = sum(1 for v in part_category.values() if v == "Repeater")
stranger_count = sum(1 for v in part_category.values() if v == "Stranger")
other_count    = len(part_category) - runner_count - repeater_count - stranger_count
print(f"  Part categories — Runner: {runner_count}  "
      f"Repeater: {repeater_count}  Stranger: {stranger_count}"
      + (f"  Other/unknown: {other_count}" if other_count > 0 else ""))

# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"  Machine state loaded  :")
        for m, p in state.items():
            print(f"    {m:25s} last ran → {p}")
        return state
    print("  Machine state         : FIRST RUN — no previous state")
    print("                          No changeover penalties applied today.")
    print(f"                          State will be saved to '{MACHINE_STATE_FILE}'")
    return {}

def save_machine_state(hz_state, vt_state):
    combined = {**hz_state, **vt_state}
    combined = {m: p for m, p in combined.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → {MACHINE_STATE_FILE}")
    for m, p in combined.items():
        print(f"    {m:25s} last ran → {p}")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

hz_compat, hz_machines = build_compatibility(hz_matrix)
vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — INDENT HORIZON TABLE
# -------------------------------------------------------------
# Replaces the old demand horizon.  For every part shows:
#
#   Monthly_Indent      : full month target from HZ/VT sheet
#   Working_Days        : denominator used (auto-calculated)
#   Daily_Indent        : Monthly_Indent / Working_Days
#   Inventory_Now       : current stock
#   Today_Target_Qty    : max(0, daily_indent - inventory)
#   Today_Target_Hrs    : today_target_qty / rate
#   Indent_Status       : INV_SUFFICIENT / PRODUCTION_NEEDED
# =============================================================

def compute_indent_horizon(parts, rate_dict):
    rows = []
    for p in parts:
        inv       = inventory.get(p, 0.0)
        monthly   = indent_monthly.get(p, 0.0)
        daily     = indent_daily.get(p, 0.0)
        target    = today_target_qty.get(p, 0.0)
        r         = rate_dict.get(p, 1.0)

        if target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":               p,
            "Monthly_Indent":     round(monthly, 0),
            "Working_Days":       WORKING_DAYS,
            "Daily_Indent":       round(daily, 2),
            "Inventory_Now":      round(inv, 0),
            "Today_Target_Qty":   round(target, 0),
            "Today_Target_Hrs":   round(target / r if r > 0 else 0, 2),
            "Indent_Status":      status,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 11 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv    = inventory.get(p, 0)
        daily  = indent_daily.get(p, 0)
        if daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — All parts have zero indent today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, "SCENARIO 0 — ALL parts critical (inv < 1 day indent)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical (inv < 1 day indent)"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, f"SCENARIO 3 — All parts healthy (≥{TARGET_DAYS_INV} days inv)"

# =============================================================
# SECTION 12 — PRIORITY SCORING
# -------------------------------------------------------------
# Priority is now based on indent coverage, not demand coverage.
#   days_coverage = inventory / daily_indent
#   urgency       = 1.0 if < 1 day, 0.5 if < 2 days, else 0
#   cv            = std/mean (variability risk — kept from V4)
#   score         = W_COVERAGE/coverage + W_URGENCY*urgency + W_RISK*cv
# =============================================================

def compute_priority(parts, horizon_df):
    horizon = horizon_df.set_index("Part")
    rows    = []

    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)

        days_cov = inv / daily if daily > 0 else 999

        if days_cov < 1:
            urgency = 1.0
        elif days_cov < 2:
            urgency = 0.5
        else:
            urgency = 0.0

        # Coefficient of variation not available without history — default 0
        cv = 0.0

        score = (W_COVERAGE * (1.0 / (days_cov + 0.01))
               + W_URGENCY  * urgency
               + W_RISK     * cv)

        target = horizon.loc[p, "Today_Target_Qty"] if p in horizon.index else 0

        rows.append({
            "Part":            p,
            "Days_Coverage":   round(days_cov, 2),
            "Urgency":         urgency,
            "CV":              round(cv, 3),
            "Score":           round(score, 4),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Today_Target":    round(target, 0),
        })

    return (pd.DataFrame(rows)
              .sort_values("Score", ascending=False)
              .reset_index(drop=True))

# =============================================================
# SECTION 13 — MACHINE RANKER  (category-aware, unchanged)
# =============================================================

def rank_machines(part, machines, compatibility, machine_hours,
                  machine_last_part, changeover_dict, inv_days):
    category    = part_category.get(part, "Stranger")
    is_runner   = (category == "Runner")
    runner_lock = is_runner and inv_days <= 1.0

    ranked = []
    for m in machines:
        if m not in compatibility.get(part, []):
            continue

        used = machine_hours.get(m, 0)
        free = AVAILABLE_HOURS - used
        if free < MIN_RUN_HOURS:
            continue

        last = machine_last_part.get(m)

        if runner_lock and last != part:
            continue

        co_hrs = 0.0 if last == part else changeover_dict.get(m, DEFAULT_CHANGEOVER_HRS)

        effective_free = free - co_hrs
        if effective_free < MIN_RUN_HOURS:
            continue

        cost = co_hrs / AVAILABLE_HOURS + (used / AVAILABLE_HOURS)
        ranked.append((m, cost, effective_free, co_hrs))

    ranked.sort(key=lambda x: x[1])
    return ranked, runner_lock

# =============================================================
# SECTION 14 — 22-HOUR UTILIZATION FILLER
# =============================================================

def stagger_changeovers(plan, machines, changeover_dict):
    print(f"\n  Tool-changer stagger:")

    def get_co_events(plan, machines, changeover_dict):
        events = []
        for m in machines:
            m_rows = [r for r in plan if r["Machine"] == m]
            if len(m_rows) < 2:
                continue
            cursor = 0.0
            for i, row in enumerate(m_rows):
                co_h  = float(row.get("Changeover_Hrs") or 0.0)
                run_h = float(row.get("Run_Hours") or 0.0)
                if co_h > 0:
                    events.append({
                        "machine":     m,
                        "part_before": m_rows[i-1]["Part"] if i > 0 else "—",
                        "part_after":  row["Part"],
                        "co_start":    cursor,
                        "co_duration": co_h,
                        "co_end":      cursor + co_h,
                        "row_before":  m_rows[i-1],
                        "row_after":   row,
                    })
                cursor += co_h + run_h
        events.sort(key=lambda e: e["co_start"])
        return events

    max_passes = 20
    for pass_num in range(max_passes):
        events = get_co_events(plan, machines, changeover_dict)
        if not events:
            print(f"    No changeovers in plan — nothing to stagger")
            return

        conflict_found = False
        for i in range(1, len(events)):
            prev = events[i - 1]
            curr = events[i]
            if curr["co_start"] < prev["co_end"]:
                conflict_found = True
                overlap = prev["co_end"] - curr["co_start"]

                row_before_curr = curr["row_before"]
                run_h_before    = float(row_before_curr.get("Run_Hours") or 0.0)
                r_val_before    = rate.get(row_before_curr["Part"], 1)
                lost_qty_A      = overlap * r_val_before
                feasible_A      = (run_h_before - overlap) >= MIN_RUN_HOURS

                row_before_prev = prev["row_before"]
                run_h_prev      = float(row_before_prev.get("Run_Hours") or 0.0)
                r_val_prev      = rate.get(row_before_prev["Part"], 1)
                m_prev          = prev["machine"]
                used_prev       = sum(
                    float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
                    for r in plan if r["Machine"] == m_prev
                )
                spare_prev  = AVAILABLE_HOURS - used_prev
                shift_B     = min(overlap, spare_prev)
                lost_qty_B  = 0.0 if spare_prev >= overlap else (overlap - spare_prev) * r_val_prev
                feasible_B  = shift_B > 0

                if feasible_A and feasible_B:
                    use_A = lost_qty_A <= lost_qty_B
                elif feasible_A:
                    use_A = True
                elif feasible_B:
                    use_A = False
                else:
                    print(f"    ⚠ Cannot resolve conflict — insufficient capacity to shift")
                    continue

                if use_A:
                    new_run_h = round(run_h_before - overlap, 3)
                    lost_qty  = round(overlap * r_val_before, 0)
                    row_before_curr["Run_Hours"]      = new_run_h
                    row_before_curr["Production_Qty"] = round(
                        row_before_curr["Production_Qty"] - lost_qty, 0)
                    row_before_curr["Total_Hrs_Used"] = round(
                        float(row_before_curr.get("Changeover_Hrs") or 0) + new_run_h, 3)
                    row_before_curr["Stagger_Adjusted"] = \
                        f"Reduced {round(overlap*60,1)}min — CO shifted earlier"
                    print(f"    ✓ {curr['machine']:15s}  CO conflict resolved (Option A)")
                else:
                    new_run_h  = round(run_h_prev + shift_B, 3)
                    extra_qty  = round(shift_B * r_val_prev, 0)
                    row_before_prev["Run_Hours"]      = new_run_h
                    row_before_prev["Production_Qty"] = round(
                        row_before_prev["Production_Qty"] + extra_qty, 0)
                    row_before_prev["Total_Hrs_Used"] = round(
                        float(row_before_prev.get("Changeover_Hrs") or 0) + new_run_h, 3)
                    row_before_prev["Stagger_Adjusted"] = \
                        f"Extended {round(shift_B*60,1)}min — CO shifted later"
                    print(f"    ✓ {prev['machine']:15s}  CO conflict resolved (Option B)")
                break

        if not conflict_found:
            print(f"    All changeovers staggered — no conflicts  ✓")
            break
    else:
        print(f"  ⚠ Stagger did not fully resolve in {max_passes} passes — check manually")


def fill_remaining_hours(plan, machine_hours, machine_last_part,
                         parts, compatibility, machines,
                         current_inventory, horizon_df, scenario,
                         changeover_dict):
    """
    Fill remaining machine time after primary assignments.
    Option A: extend an already-planned part (no new row).
    Option B: add a new part (only if ≥ MIN_RUN_HOURS remain and not already planned).
    """
    horizon        = horizon_df.set_index("Part")
    already_planned = {row["Part"] for row in plan}

    for m in machines:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
        if remaining <= 0:
            continue

        on_machine = [row["Part"] for row in plan if row["Machine"] == m]

        # Option A: extend existing part
        best_part, best_gap = None, -1
        for p in on_machine:
            inv_now    = current_inventory.get(p, 0)
            target_inv = indent_daily.get(p, 0) * TARGET_DAYS_INV
            gap        = max(0, target_inv - inv_now)
            if gap > best_gap:
                best_gap, best_part = gap, p

        if best_part and best_gap > 0:
            p         = best_part
            r_val     = rate.get(p, 1)
            extra_qty = round(remaining * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(row["Run_Hours"] + remaining, 2)
                    row["Production_Qty"] = round(row["Production_Qty"] + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        row.get("Total_Hrs_Used", row["Run_Hours"]) + remaining, 2)
                    row["Type"] = "Primary+Extended (inv build)"
                    break
            machine_hours[m]     += remaining
            current_inventory[p]  = current_inventory.get(p, 0) + extra_qty
            print(f"    ↑ {p:30s} → {m:15s}  +{remaining:.2f}h merged  "
                  f"qty+={extra_qty:.0f}  [EXTEND]")
            continue

        # Option B: new part
        if remaining < MIN_RUN_HOURS:
            continue

        candidates = []
        for p in parts:
            if p in already_planned:
                continue
            if m not in compatibility.get(p, []):
                continue
            inv_now    = current_inventory.get(p, 0)
            daily      = indent_daily.get(p, 0)
            tgt_inv    = daily * TARGET_DAYS_INV
            if scenario == 3 and inv_now >= tgt_inv:
                continue
            gap = max(0, tgt_inv - inv_now)
            candidates.append((p, gap))

        candidates.sort(key=lambda x: -x[1])

        for p, gap in candidates:
            last  = machine_last_part.get(m)
            co_hrs = 0.0 if last == p else changeover_dict.get(m, DEFAULT_CHANGEOVER_HRS)
            changeover_flag = "No" if co_hrs == 0 else "Yes"
            effective_run   = remaining - co_hrs
            if effective_run < MIN_RUN_HOURS:
                continue

            r_val = rate.get(p, 1)
            qty   = round(effective_run * r_val, 0)

            machine_hours[m]     += (co_hrs + effective_run)
            current_inventory[p]  = current_inventory.get(p, 0) + qty
            machine_last_part[m]  = p
            already_planned.add(p)

            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(effective_run, 2),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + effective_run, 2),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(indent_daily.get(p, 0), 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       changeover_flag,
                "Type":             "New — inv fill",
                "Runner_Lock":      "No",
                "Priority_Score":   0,
                "Stagger_Adjusted": "No",
            })
            print(f"    + {p:30s} → {m:15s}  {effective_run:.2f}h  qty={qty:.0f}  "
                  f"[New fill]  CO={changeover_flag}")
            break

# =============================================================
# SECTION 15 — MAIN SCHEDULER
# =============================================================

def schedule(parts, compatibility, machines, changeover_dict, label=""):

    print(f"\n{'─'*62}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*62}")

    scenario, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    # Indent horizon for this group
    horizon_df  = compute_indent_horizon(parts, rate)
    horizon_idx = horizon_df.set_index("Part")

    prod_needed = horizon_df[horizon_df["Indent_Status"] == "PRODUCTION NEEDED"]
    if not prod_needed.empty:
        print(f"\n  Production needed today ({len(prod_needed)} parts):")
        print(f"  {'Part':<30}  {'Monthly':>8}  {'Daily':>8}  {'Inv':>8}  {'Target':>8}")
        print(f"  {'─'*30}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*8}")
        for _, r in prod_needed.iterrows():
            print(f"  {r['Part']:<30}  "
                  f"{r['Monthly_Indent']:>8.0f}  "
                  f"{r['Daily_Indent']:>8.2f}  "
                  f"{r['Inventory_Now']:>8.0f}  "
                  f"{r['Today_Target_Qty']:>8.0f}")
    else:
        print("  No production needed — inventory covers daily indent for all parts")

    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()

    plan, deferred, not_planned = [], [], []

    priority_df = compute_priority(parts, horizon_df)

    print(f"\n  Priority order:")
    for _, r in priority_df.iterrows():
        print(f"    {r['Part']:30s}  "
              f"score={r['Score']:.3f}  "
              f"cov={r['Days_Coverage']}d  "
              f"daily_indent={r['Daily_Indent']:.2f}  "
              f"target={r['Today_Target']:.0f}")

    runner_dedicated_machines = set()

    def run_assignment(parts_subset, pass_label, exclude_machines=None):
        if exclude_machines is None:
            exclude_machines = set()
        print(f"\n  {pass_label}:")

        for _, row in priority_df[priority_df["Part"].isin(parts_subset)].iterrows():
            part     = row["Part"]

            if any(r["Part"] == part for r in plan):
                print(f"    ~ {part:30s}  ALREADY PLANNED — skipped")
                continue

            inv_now   = current_inventory.get(part, 0)
            daily_ind = indent_daily.get(part, 0)
            monthly   = indent_monthly.get(part, 0)
            r_val     = rate.get(part, 1)
            category  = part_category.get(part, "Stranger")
            inv_days  = inv_now / daily_ind if daily_ind > 0 else 999

            compatible_mch = compatibility.get(part, [])
            target         = today_target_qty.get(part, 0)

            # ── Zero indent ──────────────────────────────────
            if monthly == 0:
                deferred.append({
                    "Part":              part,
                    "Category":          category,
                    "Inventory_Now":     round(inv_now, 0),
                    "Monthly_Indent":    0,
                    "Daily_Indent":      0,
                    "Today_Target":      0,
                    "Days_Coverage":     round(inv_days, 2),
                    "Reason":            "Monthly indent is zero — no production required",
                    "Next_Action":       "Will be re-evaluated when indent appears",
                })
                print(f"    - {part:30s} [{category:8s}]  NOT REQUIRED — indent = 0")
                continue

            # ── Inventory sufficient ─────────────────────────
            if target == 0:
                deferred.append({
                    "Part":              part,
                    "Category":          category,
                    "Inventory_Now":     round(inv_now, 0),
                    "Monthly_Indent":    round(monthly, 0),
                    "Daily_Indent":      round(daily_ind, 2),
                    "Today_Target":      0,
                    "Days_Coverage":     round(inv_days, 2),
                    "Reason":            "Inventory covers daily indent — no production needed today",
                    "Next_Action":       "Re-evaluate tomorrow",
                })
                print(f"    - {part:30s} [{category:8s}]  NOT REQUIRED  "
                      f"inv={inv_now:.0f} ≥ daily_indent={daily_ind:.2f}")
                continue

            # ── Target hours ─────────────────────────────────
            if category == "Runner":
                # Cap at demand hours + 1-day buffer to avoid over-run
                target_hrs = min(
                    (target / r_val if r_val > 0 else MIN_RUN_HOURS) + (daily_ind / r_val if r_val > 0 else 0),
                    AVAILABLE_HOURS
                )
                target_hrs = max(target_hrs, MIN_RUN_HOURS)
            else:
                target_hrs = max(
                    MIN_RUN_HOURS,
                    target / r_val if r_val > 0 else MIN_RUN_HOURS
                )

            # ── No compatible machines at all ────────────────
            if not compatible_mch:
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily_ind, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": "NONE DEFINED",
                    "Reason":              "Part has no compatible machines in the matrix",
                    "Action_Needed":       "Add this part to the compatibility matrix",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — not in matrix")
                continue

            # ── Available machines for this pass ─────────────
            available_mch = [m for m in compatible_mch if m not in exclude_machines]
            if not available_mch:
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily_ind, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              (
                        f"All compatible machines dedicated to Runner parts today "
                        f"({', '.join(exclude_machines & set(compatible_mch))})."
                    ),
                    "Action_Needed":       "Add more compatible machines for this part in matrix",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — all machines Runner-dedicated")
                continue

            # ── Rank machines ─────────────────────────────────
            ranked, runner_lock = rank_machines(
                part, available_mch, compatibility,
                machine_hours, machine_last_part,
                changeover_dict, inv_days
            )

            if not ranked:
                machines_status = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                                   for m in available_mch if m in machine_hours}
                not_in_group    = [m for m in available_mch if m not in machine_hours]

                if runner_lock:
                    last_machine = next(
                        (m for m in available_mch if machine_last_part.get(m) == part), None)
                    if last_machine:
                        free_on_last = round(AVAILABLE_HOURS - machine_hours.get(last_machine, 0), 2)
                        reason  = (f"RUNNER (inv={round(inv_days,2)}d ≤ 1d) — must stay on "
                                   f"{last_machine} but only {free_on_last}h free")
                        action  = f"Free capacity on {last_machine} or check inventory covers gap"
                    else:
                        reason  = f"RUNNER (inv={round(inv_days,2)}d ≤ 1d) — no machine state"
                        action  = "Machine state saves after this run — enforces from tomorrow"
                elif not_in_group:
                    reason  = f"Compatible machines {not_in_group} not in this machine group"
                    action  = "Check compatibility matrix — wrong group (HZ/VT)?"
                elif machines_status and all(h < MIN_RUN_HOURS for h in machines_status.values()):
                    reason  = ("All compatible machines fully utilised. Free: "
                               + ", ".join(f"{m}={h}h" for m, h in machines_status.items()))
                    action  = "Reduce lower-priority part qtys or plan tomorrow"
                else:
                    reason  = ("Remaining hrs < MIN_RUN_HOURS after changeover. Free: "
                               + ", ".join(f"{m}={h}h" for m, h in machines_status.items()))
                    action  = "Inventory must cover gap — check Inventory_Health sheet"

                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily_ind, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              reason,
                    "Action_Needed":       action,
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — {reason[:60]}")
                continue

            # ── Assign ───────────────────────────────────────
            assigned = False
            for m, cost, effective_free, co_hrs in ranked:
                run_h = min(target_hrs, effective_free)
                run_h = max(run_h, MIN_RUN_HOURS)
                run_h = min(run_h, effective_free)
                qty   = round(run_h * r_val, 0)

                machine_hours[m]        += (co_hrs + run_h)
                current_inventory[part]  = current_inventory.get(part, 0) + qty
                machine_last_part[m]     = part

                if category == "Runner":
                    runner_dedicated_machines.add(m)

                plan.append({
                    "Part":              part,
                    "Category":          category,
                    "Machine":           m,
                    "Run_Hours":         round(run_h, 2),
                    "Changeover_Hrs":    round(co_hrs, 3),
                    "Total_Hrs_Used":    round(co_hrs + run_h, 2),
                    "Rate_Per_Hour":     round(r_val, 2),
                    "Production_Qty":    qty,
                    "Monthly_Indent":    round(monthly, 0),
                    "Daily_Indent":      round(daily_ind, 2),
                    "Today_Target":      round(target, 0),
                    "Changeover":        "No" if co_hrs == 0 else "Yes",
                    "Type":              "Primary",
                    "Runner_Lock":       "YES" if runner_lock else "No",
                    "Priority_Score":    row["Score"],
                    "Stagger_Adjusted":  "No",
                })
                co_str  = "No" if co_hrs == 0 else f"Yes ({co_hrs*60:.0f}min)"
                ded_str = "  [DEDICATED]" if category == "Runner" else ""
                print(f"    ✓ {part:30s} [{category:8s}] → {m:15s}  "
                      f"{run_h:.2f}h  qty={qty:>8.0f}  CO={co_str}{ded_str}")
                assigned = True
                break

            if not assigned:
                machines_status = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                                   for m in available_mch if m in machine_hours}
                reason = (
                    f"No single compatible machine has enough free hours "
                    f"(need {round(target_hrs,2)}h after changeover). Free: "
                    + ", ".join(f"{m}={h}h" for m, h in machines_status.items())
                )
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily_ind, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              reason,
                    "Action_Needed":       (
                        "Reduce lower-priority part qtys or verify inventory covers shortfall."
                    ),
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — "
                      f"no single machine has {round(target_hrs,2)}h free")

    # PASS 1: Runners
    runner_parts     = [p for p in parts if part_category.get(p, "Stranger") == "Runner"]
    non_runner_parts = [p for p in parts if part_category.get(p, "Stranger") != "Runner"]
    run_assignment(runner_parts,
                   "PASS 1 — Runners (dedicated machine, cap = target + 1-day buffer)")

    print(f"\n  Runner-dedicated machines: "
          f"{sorted(runner_dedicated_machines) if runner_dedicated_machines else 'none'}")

    # PASS 2: Repeaters + Strangers
    run_assignment(non_runner_parts,
                   "PASS 2 — Repeaters + Strangers (Runner-dedicated machines excluded)",
                   exclude_machines=runner_dedicated_machines)

    # 22-hour filler
    print(f"\n  22-hour filler:")
    fill_remaining_hours(
        plan, machine_hours, machine_last_part,
        list(parts), compatibility, machines,
        current_inventory, horizon_df, scenario,
        changeover_dict
    )

    # Duplicate check
    part_counts = {}
    for row in plan:
        part_counts[row["Part"]] = part_counts.get(row["Part"], 0) + 1
    duplicates = {p: c for p, c in part_counts.items() if c > 1}
    if duplicates:
        print(f"  WARNING: duplicate parts detected: {duplicates}")
    else:
        print(f"    No duplicate parts — each part appears exactly once  ✓")

    # Tool-changer stagger
    stagger_changeovers(plan, machines, changeover_dict)

    # ── Inventory health after today's plan ──────────────────
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)

        # After today: inventory + produced (supply happens next day in this model)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0

        inv_rows.append({
            "Part":               p,
            "Monthly_Indent":     round(monthly, 0),
            "Daily_Indent":       round(daily, 2),
            "Inv_Before":         round(inv_b, 0),
            "Produced_Today":     round(produced, 0),
            "Inv_After_Today":    round(inv_after, 0),
            "Days_Coverage":      round(days_cov, 2),
            "Status":             ("OK"       if days_cov >= TARGET_DAYS_INV
                                   else "LOW"      if days_cov >= 1
                                   else "CRITICAL"),
        })

    # Machine utilization
    mach_rows = []
    for m in machines:
        used      = machine_hours.get(m, 0)
        remaining = AVAILABLE_HOURS - used
        parts_run = [r["Part"] for r in plan if r["Machine"] == m]
        parts_str = ", ".join(parts_run) if parts_run else "— idle —"
        co_count  = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")

        if used >= AVAILABLE_HOURS - 0.5:
            util_status = "FULL"
        elif used >= AVAILABLE_HOURS * 0.85:
            util_status = "GOOD"
        elif used >= AVAILABLE_HOURS * 0.5:
            util_status = "PARTIAL"
        else:
            util_status = "UNDERUSED"

        mach_rows.append({
            "Machine":              m,
            "Total_Available_Hrs":  AVAILABLE_HOURS,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(remaining, 2),
            "Utilization_%":        round(used / AVAILABLE_HOURS * 100, 1),
            "Status":               util_status,
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": parts_str,
        })

    plan_df = pd.DataFrame(plan)        if plan        else pd.DataFrame()
    def_df  = pd.DataFrame(deferred)    if deferred    else pd.DataFrame()
    not_df  = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df = pd.DataFrame(mach_rows)
    inv_df  = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df["Category"] = plan_df["Part"].map(
            lambda p: part_category.get(p, "Stranger"))
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    return plan_df, def_df, not_df, mach_df, inv_df, machine_last_part, horizon_df

# =============================================================
# SECTION 16 — RUN BOTH MACHINE GROUPS
# =============================================================

hz_parts = data[data["Material"].isin(hz_matrix["Part"])]["Material"].unique()
vt_parts = data[data["Material"].isin(vt_matrix["Part"])]["Material"].unique()

hz_plan, hz_def, hz_not, hz_mach, hz_inv, hz_state, hz_horizon = \
    schedule(hz_parts, hz_compat, hz_machines, hz_changeover, "HZ Machines")

vt_plan, vt_def, vt_not, vt_mach, vt_inv, vt_state, vt_horizon = \
    schedule(vt_parts, vt_compat, vt_machines, vt_changeover, "VT Machines")

save_machine_state(hz_state, vt_state)

# =============================================================
# SECTION 17 — SAVE OUTPUT  (formatted Excel)
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "HZ_Plan_By_Machine":    "0D6E6E",
    "VT_Plan_By_Machine":    "0D6E6E",
    "HZ_Plan":               "1F4E79",
    "VT_Plan":               "1F4E79",
    "HZ_Machine_Util":       "375623",
    "VT_Machine_Util":       "375623",
    "HZ_Not_Planned":        "7B2C2C",
    "VT_Not_Planned":        "7B2C2C",
    "HZ_Not_Required_Today": "7F6000",
    "VT_Not_Required_Today": "7F6000",
    "HZ_Inventory_Health":   "4A235A",
    "VT_Inventory_Health":   "4A235A",
    "HZ_Indent_Horizon":     "154360",
    "VT_Indent_Horizon":     "154360",
    "ALL_Plan_Combined":     "1F4E79",
}

STATUS_FILLS = {
    "FULL":               PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":               PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":            PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":          PatternFill("solid", fgColor="FFC7CE"),
    "OK":                 PatternFill("solid", fgColor="C6EFCE"),
    "LOW":                PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":           PatternFill("solid", fgColor="FFC7CE"),
    "PRODUCTION NEEDED":  PatternFill("solid", fgColor="FFC7CE"),
    "INV SUFFICIENT":     PatternFill("solid", fgColor="C6EFCE"),
    "NO INDENT":          PatternFill("solid", fgColor="EDEDED"),
}

def style_sheet(ws, header_hex):
    header_fill = PatternFill("solid", fgColor=header_hex)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    center      = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center
    ws.row_dimensions[1].height = 32

    for col in ws.columns:
        max_len    = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                max_len = max(max_len, len(str(cell.value)) if cell.value is not None else 0)
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 3))

    header_row = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(header_row, start=1):
        if col_name and ("Status" in str(col_name) or "Indent_Status" in str(col_name)):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value), None)
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def build_machine_wise_plan(plan_df, machines, changeover_dict, label):
    if plan_df.empty:
        return pd.DataFrame()

    def safe_float(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    def fmt_time(h):
        try:
            h         = safe_float(h, 0.0)
            total_min = int(round(h * 60))
            return f"{total_min // 60:02d}:{total_min % 60:02d}"
        except Exception:
            return "??"

    rows = []
    for m in machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        cumulative_hrs = 0.0
        seq            = 1

        for _, pr in machine_rows.iterrows():
            part  = pr.get("Part", "—")
            run_h = safe_float(pr.get("Run_Hours", 0))
            co_h  = safe_float(pr.get("Changeover_Hrs", 0))
            co_mins = round(co_h * 60, 1)
            start_h = cumulative_hrs + co_h
            end_h   = start_h + run_h

            rows.append({
                "Machine":                m,
                "Seq":                    seq,
                "Part":                   part,
                "Category":               part_category.get(part, "Stranger"),
                "Changeover_Before_Mins": co_mins,
                "Run_Hours":              round(run_h, 2),
                "Start_Time":             fmt_time(cumulative_hrs),
                "Start_After_CO":         fmt_time(start_h),
                "End_Time":               fmt_time(end_h),
                "Cumulative_Hrs":         round(end_h, 2),
                "Production_Qty":         safe_float(pr.get("Production_Qty", 0)),
                "Monthly_Indent":         safe_float(pr.get("Monthly_Indent", 0)),
                "Daily_Indent":           safe_float(pr.get("Daily_Indent", 0)),
                "Today_Target":           safe_float(pr.get("Today_Target", 0)),
                "Changeover":             pr.get("Changeover", "No") or "No",
                "Type":                   pr.get("Type", "Primary") or "Primary",
                "Runner_Lock":            pr.get("Runner_Lock", "No") or "No",
                "Row_Type":               "Part",
            })
            cumulative_hrs = end_h
            seq           += 1

        total_used     = round(cumulative_hrs, 2)
        total_unused   = round(AVAILABLE_HOURS - total_used, 2)
        co_hrs_series  = machine_rows["Changeover_Hrs"].apply(lambda x: safe_float(x, 0))
        total_co_mins  = round(co_hrs_series.sum() * 60, 1)
        total_prod_hrs = round(total_used - co_hrs_series.sum(), 2)
        total_qty      = machine_rows["Production_Qty"].apply(lambda x: safe_float(x, 0)).sum()
        co_count       = int(machine_rows["Changeover"].eq("Yes").sum())

        rows.append({
            "Machine":                m,
            "Seq":                    "—",
            "Part":                   f"TOTAL — {m}",
            "Category":               "—",
            "Changeover_Before_Mins": total_co_mins,
            "Run_Hours":              total_prod_hrs,
            "Start_Time":             "00:00",
            "Start_After_CO":         "—",
            "End_Time":               fmt_time(total_used),
            "Cumulative_Hrs":         total_used,
            "Production_Qty":         round(total_qty, 0),
            "Monthly_Indent":         "—",
            "Daily_Indent":           "—",
            "Today_Target":           "—",
            "Changeover":             f"{co_count} changeovers",
            "Type":                   f"Used {total_used}h / {AVAILABLE_HOURS}h  |  Unused {total_unused}h",
            "Runner_Lock":            "—",
            "Row_Type":               "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    header_font  = Font(bold=True, color="FFFFFF", size=11)
    summary_fill = PatternFill("solid", fgColor="0D9488")
    summary_font = Font(bold=True, color="FFFFFF", size=11)
    part_fills   = [
        PatternFill("solid", fgColor="EFF6FF"),
        PatternFill("solid", fgColor="F0FDF4"),
    ]
    co_fill = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers       = [cell.value for cell in ws[1]]
    row_type_col  = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col        = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col   = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None

    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col - 1].value if row_type_col else ""
        machine  = row[machine_col  - 1].value if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = summary_font
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            bg = part_fills[machine_color_idx]
            for cell in row:
                cell.fill      = bg
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col - 1].value == "Yes":
                row[co_col - 1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(40, max_len + 3))
    ws.freeze_panes = "C2"


# =============================================================
# SECTION 17A — WRITE EXCEL
# =============================================================

print(f"\nWriting → {output_path}")

all_plan = pd.concat([hz_plan, vt_plan], ignore_index=True)
hz_mw    = build_machine_wise_plan(hz_plan, hz_machines, hz_changeover, "HZ")
vt_mw    = build_machine_wise_plan(vt_plan, vt_machines, vt_changeover, "VT")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    hz_mw.to_excel(writer,     sheet_name="HZ_Plan_By_Machine",    index=False)
    vt_mw.to_excel(writer,     sheet_name="VT_Plan_By_Machine",    index=False)
    hz_plan.to_excel(writer,   sheet_name="HZ_Plan",               index=False)
    vt_plan.to_excel(writer,   sheet_name="VT_Plan",               index=False)
    hz_mach.to_excel(writer,   sheet_name="HZ_Machine_Util",       index=False)
    vt_mach.to_excel(writer,   sheet_name="VT_Machine_Util",       index=False)
    hz_not.to_excel(writer,    sheet_name="HZ_Not_Planned",        index=False)
    vt_not.to_excel(writer,    sheet_name="VT_Not_Planned",        index=False)
    hz_def.to_excel(writer,    sheet_name="HZ_Not_Required_Today", index=False)
    vt_def.to_excel(writer,    sheet_name="VT_Not_Required_Today", index=False)
    hz_inv.to_excel(writer,    sheet_name="HZ_Inventory_Health",   index=False)
    vt_inv.to_excel(writer,    sheet_name="VT_Inventory_Health",   index=False)
    hz_horizon.to_excel(writer,sheet_name="HZ_Indent_Horizon",     index=False)
    vt_horizon.to_excel(writer,sheet_name="VT_Indent_Horizon",     index=False)
    all_plan.to_excel(writer,  sheet_name="ALL_Plan_Combined",     index=False)

wb = load_workbook(output_path)

for mw_sheet in ["HZ_Plan_By_Machine", "VT_Plan_By_Machine"]:
    if mw_sheet in wb.sheetnames:
        style_machine_wise_sheet(wb[mw_sheet])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and "By_Machine" not in sheet_name:
        style_sheet(wb[sheet_name], header_hex)

tab_order = [
    "HZ_Plan_By_Machine", "VT_Plan_By_Machine",
    "HZ_Plan", "VT_Plan",
    "HZ_Machine_Util", "VT_Machine_Util",
    "HZ_Not_Planned", "VT_Not_Planned",
    "HZ_Not_Required_Today", "VT_Not_Required_Today",
    "HZ_Inventory_Health", "VT_Inventory_Health",
    "HZ_Indent_Horizon", "VT_Indent_Horizon",
    "ALL_Plan_Combined",
]
for name in tab_order:
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = (
            "0D6E6E" if "By_Machine"        in name else
            "1F4E79" if "Plan"              in name else
            "375623" if "Machine"           in name else
            "7B2C2C" if "Not_Planned"        in name else
            "7F6000" if "Not_Required_Today" in name else
            "4A235A"
        )

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 18 — SUMMARY PRINT
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V5 Complete  —  {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"{'='*62}")
print(f"  HZ  planned={len(hz_plan):>4}  not_required_today={len(hz_def):>4}  not_planned={len(hz_not):>4}")
print(f"  VT  planned={len(vt_plan):>4}  not_required_today={len(vt_def):>4}  not_planned={len(vt_not):>4}")
print(f"\n  not_required_today = inventory already covers daily indent, no action needed")
print(f"  not_planned        = production needed but no machine capacity available")

print(f"\n  Output  → {output_path}")
print(f"  State   → {MACHINE_STATE_FILE}")
print(f"\n  NOTE: Update SECTION 1 each morning:")
print(f"        PLANNING_DATE = date(2026, 3, 15)")
print(f"        INDENT_MONTH  = date(2026, 3,  1)   ← only change when month rolls over")